# AI Attitudes Survey - Example Hypothesis Tests

This notebook contains example hypothesis tests using data from our AI Attitudes survey. Examples include:

1. Testing for a significant difference in population proportions.
2. Testing for a signficant difference in population means.

## Preliminaries

Run the code cell to load and clean the data and import necessary libraries. This code renames columns and drops unneeded columns.

Run this code **before** running any of the examples in this notebook.

In [ ]:
from datetime import date
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from textblob import TextBlob, Word
from wordcloud import WordCloud

file = "/home/shared/AI_Attitudes.csv"
survey = pd.read_csv(file)

# Rename and drop columns
survey.columns=['ID', 'Start', 'End', 'Email', 'Name', 'Last Edit', 'Age', 'Gender', 'Excited', 'Why Excited', 'Concerned', 'Why Concerned', 'Use Freq', 'How Used', 'Tools', 'Report']
survey.drop(columns=['Email', 'Name', 'Last Edit', 'Report'], inplace=True)
survey

# Some recoding using np.select()
# Firstly, create a Net Excited-Concerned score (Net E-C) based on Excited - Concerned
# Then recode age into ascending numerical values so we can order output later
survey['Net E-C'] = survey['Excited'] - survey['Concerned']
conditions = [survey['Net E-C']>0, survey['Net E-C']==0, survey['Net E-C']<0]
values = ['More excited than concerned', 'Equally excited and concerned', 'More concerned than excited']
survey['Net E-C Cat'] = np.select(conditions, values, default='')
conditions = [survey['Age']=='Under 18', survey['Age']=='18-24', survey['Age']=='25-34', survey['Age']=='35-44', survey['Age']=='45-54', survey['Age']=='55 or over', survey['Age']=='Prefer not to say']
values = [0, 1, 2, 3, 4, 5, 6]
survey['Age Code'] = np.select(conditions, values, default=0)
survey

## 1. Testing for Significant Difference in Proportions

In this example we compare the proportion of males and females who reported to be 'more excited than concerned' based on their net excited-concerned score.

We are interested in whether any difference in proportions for these two groups is statistically significant. We will use a 5% significance level, which corresponds to a 95% confidence level.

We will use $p_m$ to represent the proportion of males who reported being more excited than concerned. We will use $p_f$ to represent the proportion of males who reported being more excited than concerned.

Our hypotheses are:

$$ H_0: p_m = p_f$$
$$ H_1: p_m \ne p_f$$

Significance level: $\alpha = 0.05$

In [ ]:
# First compare counts for males and females
print("Male")
print(survey[survey['Gender']=='Male']['Net E-C Cat'].value_counts())
print("Female")
print(survey[survey['Gender']=='Female']['Net E-C Cat'].value_counts())

In [ ]:
# Now conduct the hypothesis test
from statsmodels.stats.proportion import proportions_ztest

more_excited_m = 20
more_excited_f = 37
total_m = 50
total_f = 61

print(f"Percent of males more excited than concerned: {more_excited_m/total_m*100:.1f}%")
print(f"Percent of females more excited than concerned: {more_excited_f/total_f*100:.1f}%")

count = np.array([more_excited_m, more_excited_f])
nobs = np.array([total_m,total_f])

# Perform the z-test for two proportions
z_stat, p_value = proportions_ztest(count=count, nobs=nobs, alternative='two-sided')

print(f"Z-statistic: {z_stat:.4f}")
print(f"P-value: {p_value:.4f}")

# Decision
if p_value < 0.05:
    print("Reject the null hypothesis: Significant difference found at the 95% confidence level.")
else:
    print("Fail to reject the null hypothesis: No significant difference found at the 95% confidence level.")

## 2. Testing for Significant Difference in Means

In this example we compare the mean 'excited' score for those aged under 35 with all others. 

We are interested in whether any difference in the mean 'excited score' for these two groups is statistically significant. We will use a 5% significance level, which corresponds to a 95% confidence level.

We will use $\mu_1$ to represent the mean excited score for those under 35. We will use $\mu_2$ to represent the mean excited score for others.

Our hypotheses are:

$$ H_0: \mu_1 = \mu_2$$
$$ H_1: \mu_1 \ne \mu_2$$

Significance level: $\alpha = 0.05$

In [ ]:
# Let's print the means() to see if they are similar
print(f"Mean excited score for those under 35: {survey[survey['Age Code'] < 2]['Excited'].mean():.2f}")
print(f"Mean excited score for others: {survey[survey['Age Code'] >= 2]['Excited'].mean():.2f}")

In [ ]:
from scipy import stats

t1 = survey[survey['Age Code'] < 2]['Excited']
t2 = survey[survey['Age Code'] >= 2]['Excited']

# Perform paired samples t-test
t_statistic_rel, p_value_rel = stats.ttest_ind(t1, t2)

print(f"Independent Samples T-test Results")
print(f"T-statistic: {t_statistic_rel:.4f}")
print(f"P-value: {p_value_rel:.4f}\n")
if p_value_rel < 0.05:
    print(f"Since {p_value_rel:.4f} < 0.05, the difference between the means is statistically significant at the 95% confidence level.")
else:
    print(f"Since {p_value_rel:.4f} >= 0.05, the difference between the means is NOT statistically significant at the 95% confidence level.")